# Lesion Segmentation
Segments the lesion from the background in each dermoscopy image using a colour-based approach (LAB colour space + Otsu thresholding), crops to the lesion bounding box, and saves the processed images to `segmented_cache/`.

**Run after** `prepare_data.ipynb` — requires `dataset_manifest.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
!pip install scikit-image scipy opencv-python-headless -q

In [ ]:
from pathlib import Path

# ── Edit these paths to match your Drive layout ────────────────────────────────
DRIVE_ROOT     = Path('/content/drive/MyDrive/PanDerm_Scripts')
OUTPUT_DIR     = DRIVE_ROOT / 'results'           # contains dataset_manifest.csv
SEGMENTED_DIR  = DRIVE_ROOT / 'segmented_cache'   # output folder
# ───────────────────────────────────────────────────────────────────────────────

IMAGE_SIZE        = 224
MORPH_KERNEL_SIZE = 15
MIN_LESION_RATIO  = 0.01
CROP_MARGIN       = 0.1

SEGMENTED_DIR.mkdir(parents=True, exist_ok=True)
print('Config ready')
print(f'  Manifest   : {OUTPUT_DIR}/dataset_manifest.csv')
print(f'  Output dir : {SEGMENTED_DIR}')

## Cell 4 — Segmentation functions

In [ ]:
import cv2
import numpy as np
from scipy import ndimage
from skimage.measure import label as sk_label
from PIL import Image

def segment_lesion(image, morph_kernel_size=15, min_lesion_ratio=0.01):
    """
    Colour-based lesion segmentation.
    1. Convert to LAB colour space
    2. Gaussian blur on L channel
    3. Otsu threshold (dark lesion = foreground)
    4. Morphological closing to fill gaps
    5. Fill holes
    6. Keep largest connected component
    7. Fallback to full image if mask too small
    """
    lab      = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l_chan   = lab[:, :, 0]
    blurred  = cv2.GaussianBlur(l_chan, (11, 11), 0)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (morph_kernel_size, morph_kernel_size)
    )
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    filled = ndimage.binary_fill_holes(closed).astype(np.uint8)

    # Keep largest connected component
    labeled = sk_label(filled, background=0)
    mask = filled
    if labeled.max() > 0:
        largest_label = max(range(1, labeled.max() + 1),
                            key=lambda l: (labeled == l).sum())
        mask = (labeled == largest_label).astype(np.uint8)
        mask = ndimage.binary_fill_holes(mask).astype(np.uint8)

    # Fallback: mask too small
    total_area = mask.shape[0] * mask.shape[1]
    if mask.sum() < min_lesion_ratio * total_area:
        mask = np.ones_like(mask, dtype=np.uint8)

    return mask


def apply_mask_and_crop(image, mask, target_size=224, margin=0.1):
    """
    Crop image to lesion bounding box with margin, apply mask, resize.
    """
    h, w = mask.shape
    ys, xs = np.where(mask > 0)

    if len(ys) == 0:
        return cv2.resize(image, (target_size, target_size), interpolation=cv2.INTER_CUBIC)

    y_min, y_max = ys.min(), ys.max()
    x_min, x_max = xs.min(), xs.max()

    pad_y = int((y_max - y_min) * margin)
    pad_x = int((x_max - x_min) * margin)
    y_min = max(0, y_min - pad_y)
    y_max = min(h, y_max + pad_y)
    x_min = max(0, x_min - pad_x)
    x_max = min(w, x_max + pad_x)

    cropped = image[y_min:y_max, x_min:x_max].copy()
    crop_mask = mask[y_min:y_max, x_min:x_max]
    cropped[crop_mask == 0] = 0

    return cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_CUBIC)

print('Segmentation functions loaded')

## Cell 5 — Process all images

In [ ]:
import pandas as pd

manifest_csv = OUTPUT_DIR / 'dataset_manifest.csv'
if not manifest_csv.exists():
    raise FileNotFoundError(f'Manifest not found: {manifest_csv}\nRun prepare_data.ipynb first.')

df = pd.read_csv(manifest_csv)
n_total    = len(df)
n_fallback = 0
segmented_paths = []

for idx, row in df.iterrows():
    img_path    = Path(row['image_path'])
    diagnosis   = row['diagnosis']
    folder_name = row.get('folder_name', row['patient_id'])

    out_dir  = SEGMENTED_DIR / diagnosis / str(folder_name)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / img_path.name

    # Skip already processed
    if out_path.exists():
        segmented_paths.append(str(out_path))
        continue

    try:
        pil_img = Image.open(str(img_path)).convert('RGB')
        image   = np.array(pil_img)
    except Exception as e:
        print(f'  ERROR loading {img_path}: {e}')
        segmented_paths.append(str(img_path))
        continue

    mask = segment_lesion(image, MORPH_KERNEL_SIZE, MIN_LESION_RATIO)

    if mask.sum() == mask.shape[0] * mask.shape[1]:
        n_fallback += 1

    cropped = apply_mask_and_crop(image, mask, IMAGE_SIZE, CROP_MARGIN)
    Image.fromarray(cropped).save(str(out_path), quality=95)
    segmented_paths.append(str(out_path))

    if (idx + 1) % 50 == 0:
        print(f'  [{idx + 1}/{n_total}] processed')

df['segmented_path'] = segmented_paths
print(f'\nDone: {n_total} images processed, {n_fallback} fallbacks to full image')

# Save updated manifest
df.to_csv(manifest_csv, index=False)
print(f'Manifest updated with segmented_path column: {manifest_csv}')

## Cell 6 — QC montage (visual check)

In [ ]:
import matplotlib.pyplot as plt

N_SAMPLES = 10
rng        = np.random.RandomState(42)
sample_idx = sorted(rng.choice(len(df), size=min(N_SAMPLES, len(df)), replace=False))

fig, axes = plt.subplots(N_SAMPLES, 3, figsize=(12, 3 * N_SAMPLES))

for i, idx in enumerate(sample_idx):
    row     = df.iloc[idx]
    pil_img = Image.open(row['image_path']).convert('RGB')
    image   = np.array(pil_img)
    mask    = segment_lesion(image, MORPH_KERNEL_SIZE, MIN_LESION_RATIO)
    cropped = apply_mask_and_crop(image, mask, IMAGE_SIZE, CROP_MARGIN)

    overlay = image.copy()
    overlay[mask == 1] = (overlay[mask == 1] * 0.6 + np.array([0, 255, 0]) * 0.4).astype(np.uint8)

    axes[i, 0].imshow(image);   axes[i, 0].set_title(f"{row['diagnosis']}/{row['patient_id']}", fontsize=8); axes[i, 0].axis('off')
    axes[i, 1].imshow(overlay); axes[i, 1].set_title('Mask overlay', fontsize=8);                            axes[i, 1].axis('off')
    axes[i, 2].imshow(cropped); axes[i, 2].set_title('Cropped lesion', fontsize=8);                          axes[i, 2].axis('off')

plt.suptitle('Segmentation QC Montage', fontsize=14, fontweight='bold')
plt.tight_layout()

qc_path = OUTPUT_DIR / 'segmentation_qc_montage.png'
plt.savefig(str(qc_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {qc_path}')